# Tutorial 4: Attention Heads Decoded

**Prerequisites:** T01 (Transformers), T02 (Hooks), T03 (Residual Stream)

**What you'll learn:**
- What attention heads actually compute and how to think about them
- The QK circuit ("what to attend to") and OV circuit ("what to write")
- How to identify common head types: previous-token, induction, copy, inhibition
- How to measure head importance through ablation

---

## Two Circuits in Every Head

Every attention head implements two independent computations:

1. **QK circuit** (attention pattern): "Where should I look?"
   - Computes `softmax(Q @ K^T / sqrt(d_head))` 
   - This is a matrix of weights saying how much each position attends to each other

2. **OV circuit** (output): "What should I write?"
   - Computes `attention_pattern @ V @ W_O`
   - This is what gets added to the residual stream

The key insight: **these are separate computations that can be analyzed independently**. A head that always attends to the previous token (QK circuit) could write anything (OV circuit).

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=2, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 42, 17, 42, 17])
logits, cache = model.run_with_cache(tokens)
print(f'Model ready. Tokens: {list(np.array(tokens))}')
print(f'(Note: tokens repeat — [1, 42, 17, 42, 17] — useful for studying induction.)')

## Reading Attention Patterns

The attention pattern is a `[seq, seq]` matrix (per head) where `pattern[q, k]` tells you how much position `q` attends to position `k`.

Common patterns to look for:
- **Diagonal** (attend to self): each position looks at itself
- **Previous token** (off-diagonal by 1): each position looks at the one before it
- **First token** (column 0 is bright): everything attends to the beginning
- **Induction** (matching patterns): positions attend to tokens that followed similar tokens before

In [ ]:
# Look at all attention patterns in layer 0
patterns = cache['blocks.0.attn.hook_pattern']  # [n_heads, seq, seq]

for head in range(cfg.n_heads):
    p = np.array(patterns[head]).round(2)
    print(f'Head 0.{head}:')
    print(f'  Rows = query positions, Cols = key positions')
    for row in range(p.shape[0]):
        cells = [f'{v:.2f}' for v in p[row]]
        print(f'  q{row}: [{"  ".join(cells)}]')
    
    # Simple pattern analysis
    diag_mass = float(jnp.trace(patterns[head])) / len(tokens)
    first_col_mass = float(jnp.sum(patterns[head, :, 0])) / len(tokens)
    print(f'  Self-attention (diagonal): {diag_mass:.2f}')
    print(f'  First-token attention:     {first_col_mass:.2f}')
    print()

## The QK Circuit: What Does the Head Look For?

The QK circuit is the matrix `W_Q^T @ W_K`, which determines the attention pattern before the softmax. We can analyze it using IRTK's circuit analysis tools.

The **full QK circuit** is `W_E^T @ W_Q^T @ W_K @ W_E`, which maps from source tokens to destination tokens: "how much does token X attend to token Y?"

In [ ]:
from irtk.circuits import qk_circuit, ov_circuit

# Analyze the QK circuit for each head in layer 0
for head in range(cfg.n_heads):
    qk = qk_circuit(model, layer=0, head=head)
    
    # The singular values tell us how "low rank" the attention pattern is
    # A rank-1 QK circuit means the head has one dominant mode of attention
    svd_result = qk.svd()
    s = np.array(svd_result[1])  # Singular values
    total = s.sum()
    top1_frac = s[0] / total if total > 0 else 0
    
    print(f'Head 0.{head} QK circuit:')
    print(f'  Top singular value explains {top1_frac:.1%} of total')
    print(f'  Singular values: {np.round(s[:4], 3)}')
    print()

## The OV Circuit: What Does the Head Write?

The OV circuit is `W_V @ W_O`, which determines what gets written to the residual stream when the head attends to a position. 

The **full OV circuit** is `W_E @ W_V @ W_O @ W_U`, which maps from source tokens to logits: "if the head attends to token X, how does that change the predictions?"

In [ ]:
# Analyze the OV circuit: "if this head attends to token X, what does it predict?"
W_E = model.embed.W_E
W_U = model.unembed.W_U

for head in range(cfg.n_heads):
    ov = ov_circuit(model, layer=0, head=head)
    
    # Full OV circuit: input token -> output logits
    # For a few source tokens, what does this head promote?
    print(f'Head 0.{head} OV circuit ("if I attend to token X, I promote..."):')
    
    for src_token in [1, 42, 17]:
        # token embedding -> through OV -> through unembed
        src_embed = W_E[src_token]  # [d_model]
        ov_mat = np.array(ov.AB)    # [d_model, d_model]
        output = src_embed @ ov_mat  # [d_model]
        logit_contribs = output @ np.array(W_U)  # [d_vocab]
        top3 = np.argsort(logit_contribs)[::-1][:3]
        top3_vals = [f'token {t} ({logit_contribs[t]:+.3f})' for t in top3]
        print(f'  Attend to token {src_token:2d} -> promotes: {", ".join(top3_vals)}')
    print()

## Identifying Head Types

IRTK includes automated tools for classifying what each head does.

In [ ]:
from irtk.head_analysis import (
    find_previous_token_heads, find_induction_heads,
    score_heads, classify_heads,
)

# Score heads on known patterns
scores = score_heads(model, tokens)
print('Head scores (higher = more like this pattern):')
print(f'{"Head":>8s}  {"Prev-Tok":>8s}  {"Induction":>9s}  {"Duplicate":>9s}')
for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        key_name = f'{layer}.{head}'
        prev = scores['previous_token'].get(key_name, 0.0)
        ind = scores['induction'].get(key_name, 0.0)
        dup = scores['duplicate_token'].get(key_name, 0.0)
        print(f'  L{layer}H{head}    {prev:8.3f}  {ind:9.3f}  {dup:9.3f}')

## Head Ablation: Does This Head Matter?

The ultimate test of whether a head matters for a prediction: **ablate it** (set its output to zero) and see how much the output changes.

In [ ]:
logits_base = model(tokens)
base_pred = int(jnp.argmax(logits_base[-1]))
base_logit = float(logits_base[-1, base_pred])

print(f'Base prediction: token {base_pred} (logit = {base_logit:.4f})')
print()
print('Effect of zeroing each head (change in logit for predicted token):')

for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        # Zero out this specific head's output
        # hook_z is [seq, n_heads, d_head] — zero out one head
        def ablate_head(z, name, h=head):
            return z.at[:, h, :].set(0.0)
        
        hook_name = f'blocks.{layer}.attn.hook_z'
        hooks = {hook_name: ablate_head}
        logits_abl = model.run_with_hooks(tokens, fwd_hooks=hooks)
        
        new_logit = float(logits_abl[-1, base_pred])
        change = new_logit - base_logit
        sign = '+' if change > 0 else '-'
        bar = sign * min(int(abs(change) * 5), 20)
        print(f'  L{layer}H{head}: {change:+.4f} {bar}')

print(f'\nNegative = head was promoting the prediction (removing it hurts).')
print(f'Positive = head was suppressing the prediction (removing it helps).')

## Composition: Heads Talking to Heads

A crucial idea: **later heads can read the outputs of earlier heads**. This is called **composition**.

There are three types of composition:
- **Q-composition**: Head B uses Head A's output to form its queries
- **K-composition**: Head B uses Head A's output to form its keys
- **V-composition**: Head B uses Head A's output to form its values

Composition is how transformers build complex behaviors from simple components.

In [ ]:
from irtk.circuits import composition_scores

# Compute composition scores between all pairs of heads
scores = composition_scores(model)

for comp_type in ['q_composition', 'k_composition', 'v_composition']:
    print(f'{comp_type.replace("_", " ").title()}:')
    matrix = scores[comp_type]  # [n_layers * n_heads, n_layers * n_heads]
    
    # Show scores from layer 0 heads -> layer 1 heads
    for src_head in range(cfg.n_heads):
        for dst_head in range(cfg.n_heads):
            score = float(matrix[src_head, cfg.n_heads + dst_head])
            bar = '#' * int(score * 20)
            print(f'  L0H{src_head} -> L1H{dst_head}: {score:.3f} {bar}')
    print()

print('High composition = the later head relies on the earlier head\'s output.')

## Key Takeaways

1. **Every head has two independent circuits**: QK (where to look) and OV (what to write)
2. **Attention patterns** reveal what positions each head connects
3. **The full OV circuit** (W_E @ W_V @ W_O @ W_U) tells you: "attending to token X promotes token Y"
4. **Head ablation** tests whether a head matters causally
5. **Composition** lets heads build on each other's work

Common head types:
- **Previous-token heads**: Attend to position n-1 (provide bigram information)
- **Induction heads**: Find repeated patterns ("A B ... A" → attend to B)
- **Copy/suppression heads**: Copy tokens to output or suppress them
- **Backup heads**: Compensate when other heads are ablated

**Next: [T05 — MLPs and Features](T05_mlps_and_features.ipynb)** — How knowledge is stored and retrieved in the MLP layers.